In [1]:
import os
os.chdir(r"D:\thesis")

from src.utils import set_seed
from src.data import HAM10000, make_loader, build_image_index, assert_no_synthetic_leakage

IMG = "data/raw/HAM10000/images"

# Safety check before every training run
assert_no_synthetic_leakage("data/splits/val.csv", "data/splits/test.csv")

# Build the path index once and share it across all three datasets
index = build_image_index(IMG)

set_seed(42)

train_ds = HAM10000("data/splits/train.csv", IMG, size=128, augment=False, index=index)
val_ds   = HAM10000("data/splits/val.csv",   IMG, size=128, augment=False, index=index)
test_ds  = HAM10000("data/splits/test.csv",  IMG, size=128, augment=False, index=index)

train_loader = make_loader(train_ds, batch_size=32, shuffle=True)
val_loader   = make_loader(val_ds,   batch_size=32, shuffle=False)
test_loader  = make_loader(test_ds,  batch_size=32, shuffle=False)

print(f"train {len(train_ds)}  val {len(val_ds)}  test {len(test_ds)}")

x, y = next(iter(train_loader))
print("batch:", x.shape, y.shape, "| labels present:", sorted(set(y.tolist())))

[data] no synthetic leakage in 2 partition(s)
[data] indexed 10015 images under data\raw\HAM10000\images
train 6981  val 1532  test 1502
batch: torch.Size([32, 3, 128, 128]) torch.Size([32]) | labels present: [0, 1, 2, 4, 5]


In [2]:
import pandas as pd
from src.data import CLASSES

print("Classes in the training CSV:", sorted(train_ds.df.dx.unique()))
print("Total:", len(train_ds.df.dx.unique()), "\n")

# How often does a random 32-image batch contain every class?
import torch
set_seed(99)
missing = []
for i, (x, y) in enumerate(train_loader):
    missing.append(7 - len(set(y.tolist())))
    if i == 49:
        break
print(f"Over 50 batches of 32: average {sum(missing)/len(missing):.1f} classes absent per batch")
print(f"Batches containing all 7 classes: {missing.count(0)} of 50")

Classes in the training CSV: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Total: 7 

Over 50 batches of 32: average 1.9 classes absent per batch
Batches containing all 7 classes: 3 of 50


In [3]:
from src.classifier import build_model, train_classifier

set_seed(42)
model = build_model()
model, history = train_classifier(
    model, train_loader, val_loader,
    epochs=30, lr=1e-4, weight_decay=1e-4, patience=7,
    tag="A_baseline_s42",
)

ep 00  loss 0.7903  val macroF1 0.4837  val acc 0.7787
ep 01  loss 0.3878  val macroF1 0.5410  val acc 0.8003
ep 02  loss 0.1734  val macroF1 0.5633  val acc 0.7852
ep 03  loss 0.0920  val macroF1 0.6045  val acc 0.8081
ep 04  loss 0.0664  val macroF1 0.5892  val acc 0.7977
ep 05  loss 0.0506  val macroF1 0.6158  val acc 0.8042
ep 06  loss 0.0334  val macroF1 0.6227  val acc 0.8166
ep 07  loss 0.0267  val macroF1 0.6085  val acc 0.8061
ep 08  loss 0.0264  val macroF1 0.6102  val acc 0.8022
ep 09  loss 0.0214  val macroF1 0.6014  val acc 0.8061
ep 10  loss 0.0164  val macroF1 0.5972  val acc 0.8048
ep 11  loss 0.0126  val macroF1 0.6381  val acc 0.8061
ep 12  loss 0.0203  val macroF1 0.5988  val acc 0.7996
ep 13  loss 0.0193  val macroF1 0.6035  val acc 0.7944
ep 14  loss 0.0087  val macroF1 0.6086  val acc 0.7879
ep 15  loss 0.0061  val macroF1 0.6082  val acc 0.8009
ep 16  loss 0.0089  val macroF1 0.6315  val acc 0.8061
ep 17  loss 0.0081  val macroF1 0.6457  val acc 0.8101
ep 18  los

In [4]:
from src.metrics import evaluate, save_metrics

metrics = evaluate(model, test_loader, bootstrap=True, seed=42)
save_metrics(metrics, "A_baseline_seed42", root="results/run2_local")

              precision    recall  f1-score   support

       akiec      0.556     0.673     0.609        52
         bcc      0.736     0.549     0.629        71
         bkl      0.655     0.581     0.616       167
          df      0.667     0.500     0.571        20
         mel      0.616     0.509     0.557       167
          nv      0.881     0.940     0.910      1004
        vasc      0.929     0.619     0.743        21

    accuracy                          0.814      1502
   macro avg      0.720     0.624     0.662      1502
weighted avg      0.806     0.814     0.807      1502

macro-F1 0.6622 | balanced acc 0.6245 | MEL recall 0.5090 | accuracy 0.8142 | macro-AUROC 0.9235


WindowsPath('results/run2_local/metrics/A_baseline_seed42.json')

In [5]:
b = metrics["bootstrap"]
print("Metric            value    95% CI")
for k in ["macro_f1", "melanoma_recall", "balanced_accuracy"]:
    print(f"{k:18s} {metrics[k]:.3f}   [{b[k]['lo']:.3f} – {b[k]['hi']:.3f}]")

print("\nPer-class recall:")
pc = metrics["bootstrap_per_class_recall"]
for c, v in metrics["per_class"].items():
    w = pc[c]["hi"] - pc[c]["lo"]
    print(f"  {c:6s} {v['recall']:.3f}  [{pc[c]['lo']:.3f} – {pc[c]['hi']:.3f}]  "
          f"width {w:.3f}  (n={v['support']})")

Metric            value    95% CI
macro_f1           0.662   [0.602 – 0.707]
melanoma_recall    0.509   [0.435 – 0.585]
balanced_accuracy  0.624   [0.562 – 0.678]

Per-class recall:
  akiec  0.673  [0.550 – 0.812]  width 0.263  (n=52)
  bcc    0.549  [0.431 – 0.667]  width 0.236  (n=71)
  bkl    0.581  [0.500 – 0.653]  width 0.153  (n=167)
  df     0.500  [0.273 – 0.727]  width 0.455  (n=20)
  mel    0.509  [0.435 – 0.585]  width 0.149  (n=167)
  nv     0.940  [0.926 – 0.955]  width 0.029  (n=1004)
  vasc   0.619  [0.385 – 0.824]  width 0.439  (n=21)


In [6]:
save_metrics(metrics, "A_baseline_seed42", root="results/run2_local")

              precision    recall  f1-score   support

       akiec      0.556     0.673     0.609        52
         bcc      0.736     0.549     0.629        71
         bkl      0.655     0.581     0.616       167
          df      0.667     0.500     0.571        20
         mel      0.616     0.509     0.557       167
          nv      0.881     0.940     0.910      1004
        vasc      0.929     0.619     0.743        21

    accuracy                          0.814      1502
   macro avg      0.720     0.624     0.662      1502
weighted avg      0.806     0.814     0.807      1502

macro-F1 0.6622 | balanced acc 0.6245 | MEL recall 0.5090 | accuracy 0.8142 | macro-AUROC 0.9235


WindowsPath('results/run2_local/metrics/A_baseline_seed42.json')

In [7]:
import json
from pathlib import Path

results = {42: metrics}

for seed in [43, 44]:
    print(f"\n{'='*60}\nSEED {seed}\n{'='*60}")
    set_seed(seed)

    tr_ds = HAM10000("data/splits/train.csv", IMG, size=128,
                     augment=False, index=index)
    tr_ld = make_loader(tr_ds, batch_size=32, shuffle=True)

    m, hist = train_classifier(
        build_model(), tr_ld, val_loader,
        epochs=30, lr=1e-4, weight_decay=1e-4, patience=7,
        tag=f"A_baseline_s{seed}",
    )
    r = evaluate(m, test_loader, bootstrap=True, seed=seed)
    save_metrics(r, f"A_baseline_seed{seed}", root="results/run2_local")
    results[seed] = r


SEED 43
ep 00  loss 0.7875  val macroF1 0.5128  val acc 0.7846
ep 01  loss 0.3864  val macroF1 0.5297  val acc 0.7905
ep 02  loss 0.1814  val macroF1 0.6290  val acc 0.8009
ep 03  loss 0.0941  val macroF1 0.6122  val acc 0.7898
ep 04  loss 0.0696  val macroF1 0.6274  val acc 0.8172
ep 05  loss 0.0459  val macroF1 0.6115  val acc 0.8016
ep 06  loss 0.0448  val macroF1 0.5876  val acc 0.7846
ep 07  loss 0.0332  val macroF1 0.6122  val acc 0.8016
ep 08  loss 0.0337  val macroF1 0.6068  val acc 0.8022
ep 09  loss 0.0275  val macroF1 0.6131  val acc 0.8061
Early stopping at epoch 9 (best macro-F1 0.6290)
              precision    recall  f1-score   support

       akiec      0.621     0.346     0.444        52
         bcc      0.563     0.690     0.620        71
         bkl      0.651     0.503     0.568       167
          df      0.467     0.350     0.400        20
         mel      0.592     0.461     0.519       167
          nv      0.864     0.942     0.901      1004
        vasc 

In [8]:
import numpy as np
import pandas as pd

rows = []
for metric in ["macro_f1", "melanoma_recall", "balanced_accuracy", "accuracy", "macro_auroc"]:
    vals = [results[s][metric] for s in [42, 43, 44]]
    rows.append({
        "metric": metric,
        "mean": np.mean(vals),
        "std": np.std(vals, ddof=1),
        "min": min(vals), "max": max(vals),
        "values": [round(v, 4) for v in vals],
    })
summary = pd.DataFrame(rows).round(4)
print(summary.to_string(index=False))

print("\nPer-class recall across seeds:")
from src.data import CLASSES
for c in CLASSES:
    v = [results[s]["per_class"][c]["recall"] for s in [42, 43, 44]]
    print(f"  {c:6s} mean {np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}   "
          f"[{min(v):.3f} – {max(v):.3f}]  (n={results[42]['per_class'][c]['support']})")

summary.to_csv("results/run2_local/metrics/A_baseline_summary.csv", index=False)

           metric   mean    std    min    max                   values
         macro_f1 0.6337 0.0377 0.5909 0.6622  [0.6622, 0.5909, 0.648]
  melanoma_recall 0.4591 0.0509 0.4072 0.5090  [0.509, 0.4611, 0.4072]
balanced_accuracy 0.6015 0.0370 0.5588 0.6245 [0.6245, 0.5588, 0.6213]
         accuracy 0.8051 0.0097 0.7949 0.8142 [0.8142, 0.7949, 0.8063]
      macro_auroc 0.9308 0.0064 0.9235 0.9356 [0.9235, 0.9333, 0.9356]

Per-class recall across seeds:
  akiec  mean 0.532 ± 0.168   [0.346 – 0.673]  (n=52)
  bcc    mean 0.643 ± 0.081   [0.549 – 0.690]  (n=71)
  bkl    mean 0.511 ± 0.066   [0.449 – 0.581]  (n=167)
  df     mean 0.483 ± 0.126   [0.350 – 0.600]  (n=20)
  mel    mean 0.459 ± 0.051   [0.407 – 0.509]  (n=167)
  nv     mean 0.947 ± 0.010   [0.940 – 0.959]  (n=1004)
  vasc   mean 0.635 ± 0.027   [0.619 – 0.667]  (n=21)
